# 3. Pore analysis with Zeo++

This notebook locates the TAPB–TFB CIF generated by Notebook 1, runs cofkit's Zeo++ analysis wrapper, and prints human-readable pore information. The CLI is executable; the equivalent `analyze_zeopp_pore_properties` Python workflow is included as commented code.

## Tutorial series

1. [Environment and first build](01_environment_and_first_build.ipynb)
2. [Topology, connectivity, and linkage examples](02_topologies_connectivities_and_linkages.ipynb)
3. **Pore analysis with Zeo++** (this notebook)

Prerequisites: run Notebook 1 successfully and have a Zeo++ 0.3 `network` executable. The `cofkit` Python package itself works without Zeo++; only this analysis requires the external binary.

## Locate Notebook 1's CIF

The build pipeline groups CIFs by coarse validation class, so the exact subdirectory can vary. This search deliberately looks through all classes and fails with a useful message if Notebook 1 has not produced a CIF.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
CIF_PATH="$(find out/tutorial_01_first_build/cifs -type f -name 'tapb__tfb__hcb.cif' -print -quit 2>/dev/null || true)"
if [[ -z "$CIF_PATH" ]]; then
  echo 'No TAPB–TFB CIF was found. Run Notebook 1 before continuing.' >&2
  exit 1
fi
printf 'Using CIF: %s\n' "$CIF_PATH"
sed -n '1,18p' "$CIF_PATH"

In [ ]:
# Python equivalent (commented out): locate Notebook 1's CIF.
# from pathlib import Path
# matches = sorted(Path("out/tutorial_01_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# if not matches:
#     raise FileNotFoundError("Run Notebook 1 before continuing.")
# cif_path = matches[0]
# print(cif_path)

## Configure Zeo++

If Zeo++ is already installed, point `COFKIT_ZEOPP_PATH` at its executable before launching Jupyter:

```bash
export COFKIT_ZEOPP_PATH=/absolute/path/to/zeo++-0.3/network
```

The cofkit CLI also reads this variable from the nearest `.env` file. The next optional cell builds upstream Zeo++ 0.3 under `$HOME/.local/opt/zeo++-0.3` when no configured executable is available. Skip it if you manage Zeo++ elsewhere.

In [ ]:
%%bash
set -euo pipefail
if [[ -n "${COFKIT_ZEOPP_PATH:-}" && -x "$COFKIT_ZEOPP_PATH" ]]; then
  printf 'Using configured Zeo++: %s\n' "$COFKIT_ZEOPP_PATH"
  exit 0
fi
INSTALL_PARENT="$HOME/.local/opt"
INSTALL_DIR="$INSTALL_PARENT/zeo++-0.3"
ARCHIVE="$INSTALL_PARENT/zeo++-0.3.tar.gz"
mkdir -p "$INSTALL_PARENT"
if [[ ! -x "$INSTALL_DIR/network" ]]; then
  command -v curl >/dev/null || { echo 'curl is required to download Zeo++.' >&2; exit 1; }
  command -v make >/dev/null || { echo 'A C++ build toolchain and make are required.' >&2; exit 1; }
  curl -L 'http://www.zeoplusplus.org/zeo++-0.3.tar.gz' -o "$ARCHIVE"
  tar -xzf "$ARCHIVE" -C "$INSTALL_PARENT"
  make -C "$INSTALL_DIR/voro++/src"
  make -C "$INSTALL_DIR"
fi
test -x "$INSTALL_DIR/network"
printf 'Zeo++ is ready at: %s\n' "$INSTALL_DIR/network"
printf '%s\n' 'Later cells use this default path when COFKIT_ZEOPP_PATH is unset.'

In [ ]:
# There is intentionally no Python API equivalent for compiling Zeo++.
# Zeo++ is an external C++ program; cofkit's Python API begins at the analysis step below.

## Run point-probe and finite-probe analysis

The baseline reports pore-limiting sphere sizes plus point-probe surface area and volume. The repeated `--probe-radius` values add accessibility-aware scans at illustrative radii of 1.20 Å and 1.86 Å. Adjust these radii to match the adsorbate model used in your own study.

The command intentionally omits `--json`: the terminal output is the concise human-readable report. A complete machine-readable `zeopp_report.json` is still written to disk.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
CIF_PATH="$(find out/tutorial_01_first_build/cifs -type f -name 'tapb__tfb__hcb.cif' -print -quit)"
ZEOPP_BIN="${COFKIT_ZEOPP_PATH:-$HOME/.local/opt/zeo++-0.3/network}"
test -n "$CIF_PATH" || { echo 'Run Notebook 1 first.' >&2; exit 1; }
test -x "$ZEOPP_BIN" || { echo "Zeo++ executable not found: $ZEOPP_BIN" >&2; exit 1; }
conda run -n cofkit cofkit analyze zeopp "$CIF_PATH" \
  --zeopp-path "$ZEOPP_BIN" \
  --output-dir out/tutorial_03_zeopp \
  --probe-radius 1.20 \
  --probe-radius 1.86 \
  --surface-samples-per-atom 250 \
  --volume-samples-total 5000

In [ ]:
# Python API equivalent (commented out), including a human-readable report.
# import os
# from pathlib import Path
# from cofkit import analyze_zeopp_pore_properties
#
# cif_path = next(Path("out/tutorial_01_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# zeopp_binary = os.environ.get(
#     "COFKIT_ZEOPP_PATH",
#     str(Path.home() / ".local/opt/zeo++-0.3/network"),
# )
# result = analyze_zeopp_pore_properties(
#     cif_path,
#     output_dir="out/tutorial_03_zeopp_python",
#     probe_radii=(1.20, 1.86),
#     surface_samples_per_atom=250,
#     volume_samples_total=5000,
#     zeopp_path=zeopp_binary,
# )
# basic = result.baseline.basic_pore_properties
# channels = result.baseline.point_probe_channels
# surface = result.baseline.point_probe_surface_area
# volume = result.baseline.point_probe_volume
# print(f"Input CIF: {result.input_cif}")
# print(f"Largest included sphere: {basic.largest_included_sphere:.3f} Å")
# print(f"Largest free sphere: {basic.largest_free_sphere:.3f} Å")
# print("Largest included sphere along a free path: "
#       f"{basic.largest_included_sphere_along_free_path:.3f} Å")
# print(f"Point-probe channels / pockets: {channels.n_channels} / {channels.n_pockets}")
# print(f"Point-probe accessible surface area: {surface.accessible_surface_area_a2:.3f} Å²")
# print(f"Point-probe accessible volume: {volume.accessible_volume_a3:.3f} Å³")
# for scan in result.probe_scans:
#     print(f"Probe radius {scan.settings.probe_radius:.2f} Å: {scan.status}")
#     if scan.surface_area is not None and scan.volume is not None:
#         print(f"  accessible surface area = {scan.surface_area.accessible_surface_area_a2:.3f} Å²")
#         print(f"  accessible volume fraction = {scan.volume.accessible_volume_fraction:.5f}")
#     if scan.accessibility is not None:
#         print(f"  accessible Voronoi-node fraction = {scan.accessibility.accessible_fraction}")
# print(f"Full report: {result.report_path}")

## How to read the main values

| Report field | Practical meaning |
|---|---|
| Largest included sphere | Diameter of the largest sphere that fits somewhere in the pore network |
| Largest free sphere | Diameter of the largest sphere that can traverse the periodic pore network |
| Largest included sphere along free path | Largest cavity encountered along a traversable path |
| Accessible surface area | Surface reachable by the selected probe under Zeo++'s geometric model |
| Accessible volume / fraction | Probe-accessible pore volume, either absolute or relative to the unit cell |
| Channels and pockets | Connected transport pathways versus isolated accessible regions |

All sphere sizes and probe settings in the report are in ångström-based units. Surface-area and volume estimates use Monte Carlo sampling, so increase the sample counts for production convergence studies.

## Inspect the retained report and raw outputs

cofkit preserves Zeo++ input/output artifacts and logs so the parsed report remains auditable.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT"
printf '%s\n' '--- generated Zeo++ files ---'
find out/tutorial_03_zeopp -type f -print | sort
printf '%s\n' '--- zeopp_report.json (first 140 lines) ---'
sed -n '1,140p' out/tutorial_03_zeopp/zeopp_report.json

In [ ]:
# Python equivalent (commented out): read the persisted report.
# import json
# from pathlib import Path
# report_path = Path("out/tutorial_03_zeopp/zeopp_report.json")
# report = json.loads(report_path.read_text())
# print(report["baseline"]["basic_pore_properties"])
# for scan in report["probe_scans"]:
#     print(scan["settings"]["probe_radius"], scan["status"])

## Before using the numbers

Pore descriptors are only as meaningful as the input geometry and atomic radii model. Treat this generated CIF as a tutorial input; for research calculations, optimize and validate the framework first, document the Zeo++ version and radii definitions, and check sampling convergence. A failed finite-probe scan is recorded in the JSON report without discarding successful baseline results.